In [3]:
#Data Validation & Quality Checks

In [6]:
#Ingest all sheets with pandas.read_excel().

import pandas as pd
import numpy as np
import os
from datetime import datetime

# Load all CSVs
customers = pd.read_csv("Customers.csv")
models = pd.read_csv("MobileModels.csv")
shops = pd.read_csv("ShopLocations.csv")
sales = pd.read_csv("Sales.csv")
stocks = pd.read_csv("Stocks.csv")
offers = pd.read_csv("Offers.csv")

# Store in a dictionary for easy iteration
datasets = {
    "Customers": customers,
    "MobileModels": models,
    "ShopLocations": shops,
    "Sales": sales,
    "Stocks": stocks,
    "Offers": offers
}

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [7]:
#Sales → Customers

# Orphan customer_id in Sales
orphan_customers = sales[~sales["customer_id"].isin(customers["customer_id"])]

print("Orphan customer_ids in Sales:", len(orphan_customers))
print(orphan_customers.head(10))

Orphan customer_ids in Sales: 0
Empty DataFrame
Columns: [sale_id, customer_id, model_id, shop_id, sale_date, quantity, discount, final_price, payment_mode, status]
Index: []


In [9]:
#Sales → Models

# Orphan model_id in Sales
orphan_models = sales[~sales["model_id"].isin(models["model_id"])]

print("Orphan model_ids in Sales:", len(orphan_models))
print(orphan_models.head(10))


Orphan model_ids in Sales: 0
Empty DataFrame
Columns: [sale_id, customer_id, model_id, shop_id, sale_date, quantity, discount, final_price, payment_mode, status]
Index: []


In [10]:
#Sales → Shops

# Orphan shop_id in Sales
orphan_shops = sales[~sales["shop_id"].isin(shops["shop_id"])]

print("Orphan shop_ids in Sales:", len(orphan_shops))
print(orphan_shops.head(10))

Orphan shop_ids in Sales: 0
Empty DataFrame
Columns: [sale_id, customer_id, model_id, shop_id, sale_date, quantity, discount, final_price, payment_mode, status]
Index: []


In [15]:
#Sales → Offers

# Orphan offer_id in Sales (if offer_id exists)
if "offer_id" in sales.columns:
    orphan_offers = sales[(sales["offer_id"].notna()) &
                          (~sales["offer_id"].isin(offers["offer_id"]))]

    print("Orphan offer_ids in Sales:", len(orphan_offers))
    print(orphan_offers.head(10))

In [14]:
#Stocks → Models

# Orphan model_id in Stocks
orphan_stock_models = stocks[~stocks["model_id"].isin(models["model_id"])]

print("Orphan model_ids in Stocks:", len(orphan_stock_models))
print(orphan_stock_models.head(10))

Orphan model_ids in Stocks: 0
Empty DataFrame
Columns: [stock_id, shop_id, model_id, available_units, last_updated]
Index: []


In [16]:
#Validate missing value percentage < 5% after cleaning

def missing_percentage(df, name):
    print(f"\nMissing value % for {name}:")
    percent = (df.isnull().sum() / len(df)) * 100
    print(percent)

missing_percentage(customers, "Customers")
missing_percentage(models, "Models")
missing_percentage(sales, "Sales")
missing_percentage(shops, "Shops")
missing_percentage(offers, "Offers")
missing_percentage(stocks, "Stocks")


Missing value % for Customers:
customer_id        0.000000
full_name          0.000000
email              5.183333
phone              3.900000
city               0.000000
loyalty_points    10.066667
dtype: float64

Missing value % for Models:
model_id      0.0
brand         0.0
model_name    0.0
ram_gb        0.0
storage_gb    0.0
price         0.0
dtype: float64

Missing value % for Sales:
sale_id         0.0000
customer_id     0.0000
model_id        0.0000
shop_id         0.0000
sale_date       0.0000
quantity        0.0000
discount        4.9825
final_price     4.9275
payment_mode    0.0000
status          0.0000
dtype: float64

Missing value % for Shops:
shop_id         0.0
location        0.0
manager_name    0.0
rating          0.0
dtype: float64

Missing value % for Offers:
offer_id            0.0
model_id            0.0
shop_id             0.0
discount_percent    0.0
start_date          0.0
end_date            0.0
campaign_name       0.0
dtype: float64

Missing value % for Stoc

In [17]:
customers["email"] = customers["email"].fillna("unknown@missing.com")

In [18]:
customers["loyalty_points"] = customers["loyalty_points"].fillna(0)

In [19]:
stocks["available_units"] = stocks.groupby("model_id")["available_units"].transform(
    lambda x: x.fillna(x.median())
)

In [20]:
def missing_percentage(df, name):
    print(f"\nMissing value % for {name}:")
    percent = (df.isnull().sum() / len(df)) * 100
    print(percent)

missing_percentage(customers, "Customers (After Cleaning)")
missing_percentage(stocks, "Stocks (After Cleaning)")


Missing value % for Customers (After Cleaning):
customer_id       0.0
full_name         0.0
email             0.0
phone             3.9
city              0.0
loyalty_points    0.0
dtype: float64

Missing value % for Stocks (After Cleaning):
stock_id           0.0
shop_id            0.0
model_id           0.0
available_units    0.0
last_updated       0.0
dtype: float64


In [21]:
def check_threshold(df, name, threshold=5):
    print(f"\nColumns exceeding {threshold}% missing in {name}:")
    print(((df.isnull().sum() / len(df)) * 100 > threshold))

check_threshold(customers, "Customers")
check_threshold(models, "Models")
check_threshold(sales, "Sales")
check_threshold(shops, "Shops")
check_threshold(offers, "Offers")
check_threshold(stocks, "Stocks")


Columns exceeding 5% missing in Customers:
customer_id       False
full_name         False
email             False
phone             False
city              False
loyalty_points    False
dtype: bool

Columns exceeding 5% missing in Models:
model_id      False
brand         False
model_name    False
ram_gb        False
storage_gb    False
price         False
dtype: bool

Columns exceeding 5% missing in Sales:
sale_id         False
customer_id     False
model_id        False
shop_id         False
sale_date       False
quantity        False
discount        False
final_price     False
payment_mode    False
status          False
dtype: bool

Columns exceeding 5% missing in Shops:
shop_id         False
location        False
manager_name    False
rating          False
dtype: bool

Columns exceeding 5% missing in Offers:
offer_id            False
model_id            False
shop_id             False
discount_percent    False
start_date          False
end_date            False
campaign_name     

In [22]:
#Logical consistency checks

#sale_date <= today

import pandas as pd

sales["sale_date"] = pd.to_datetime(sales["sale_date"], errors="coerce")
today = pd.Timestamp.today()

invalid_sales_dates = sales[sales["sale_date"] > today]

print("Sales with invalid future sale_date:", len(invalid_sales_dates))
print(invalid_sales_dates.head(10))

Sales with invalid future sale_date: 0
Empty DataFrame
Columns: [sale_id, customer_id, model_id, shop_id, sale_date, quantity, discount, final_price, payment_mode, status]
Index: []


In [23]:
#offers: end_date > start_date

offers["start_date"] = pd.to_datetime(offers["start_date"], errors="coerce")
offers["end_date"] = pd.to_datetime(offers["end_date"], errors="coerce")

invalid_offers = offers[offers["end_date"] <= offers["start_date"]]

print("Offers with end_date <= start_date:", len(invalid_offers))
print(invalid_offers.head(10))

Offers with end_date <= start_date: 15
    offer_id model_id   shop_id  discount_percent start_date   end_date  \
20   OFF6020  MOD2163  SHOP3069              21.8 2024-08-29 2024-04-30   
31   OFF6031  MOD2077  SHOP3005              16.7 2024-08-11 2024-02-10   
66   OFF6066  MOD2088  SHOP3040              31.4 2024-08-20 2024-01-05   
107  OFF6107  MOD2048  SHOP3002              11.4 2024-08-14 2024-01-21   
122  OFF6122  MOD2050  SHOP3041               9.3 2024-05-19 2024-01-02   
150  OFF6150  MOD2163  SHOP3056              35.3 2024-08-26 2024-07-03   
175  OFF6175  MOD2140  SHOP3050              21.8 2024-07-12 2024-04-11   
187  OFF6187  MOD2187  SHOP3044               5.8 2024-07-27 2024-05-10   
262  OFF6262  MOD2107  SHOP3011              28.5 2024-06-25 2024-05-27   
278  OFF6278  MOD2173  SHOP3069              33.5 2024-07-21 2024-05-07   

     campaign_name  
20      Flash Deal  
31       Clearance  
66    Festive Sale  
107   Festive Sale  
122     Flash Deal  
150      

In [24]:
import pandas as pd

# Convert dates to datetime (important)
offers["start_date"] = pd.to_datetime(offers["start_date"], errors="coerce")
offers["end_date"] = pd.to_datetime(offers["end_date"], errors="coerce")

# Find invalid rows
invalid_mask = offers["end_date"] <= offers["start_date"]

print("Number of invalid offer date rows BEFORE fix:", invalid_mask.sum())

# Fix: end_date = start_date + 30 days
offers.loc[invalid_mask, "end_date"] = offers.loc[invalid_mask, "start_date"] + pd.Timedelta(days=30)

print("Number of invalid offer date rows AFTER fix:",
      (offers["end_date"] <= offers["start_date"]).sum())

Number of invalid offer date rows BEFORE fix: 15
Number of invalid offer date rows AFTER fix: 0


In [25]:
offers.to_csv("Offers_Cleaned.csv", index=False)

In [ ]:
#Use Great Expectations for automated validation.

In [28]:
!pip install great_expectations

   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   -------------- ------------------------- 1.8/4.9 MB 9.1 MB/s eta 0:00:01
   ----------------------------- ---------- 3.7/4.9 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 8.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/813.6 kB ? eta -:--:--
   ---------------------------------------- 813.6/813.6 kB 8.9 MB/s eta 0:00:00

   -------- ------------------------------- 1/5 [marshmallow]
  Attempting uninstall: altair
   -------- ------------------------------- 1/5 [marshmallow]
    Found existing installation: altair 5.5.0
   -------- ------------------------------- 1/5 [marshmallow]
   ------------------------ --------------- 3/5 [altair]
    Uninstalling altair-5.5.0:
   ------------------------ --------------- 3/5 [altair]
      Successfully uninstalled altair-5.5.0
   ------------------------ --------------- 3/5 [altair]
   ------------------------ --------------